# SymTRELLIS Evaluation

This notebook rebuilds the main comparison and ablation tables directly from an evaluation workspace. It uses only Python, pandas, the workspace metadata, shape filenames, and score JSON files.

Each table is evaluated on its own largest common-success subset: a sample is retained only when every trial shown in that table has `eval_success=True`. Failed or missing evaluations are not assigned penalty values. Consequently, every method within one table is averaged over exactly the same samples, while the main and ablation tables may use different subsets.

In [ ]:
import json
from pathlib import Path

import pandas as pd

## Configuration

`model_folders` is the complete catalog of available trials. The two ordered lists below decide which trials are displayed. New sparse-structure mapper trials remain available in the catalog but are not selected in the current tables.

In [ ]:
dataworkspace_dir = Path("/mnt/scratch/eval_workspace_after_acceptance")

model_folders = {
    "TripoSG": "reference_scores/triposg_seed_43",
    "Hunyuan3D-2.1": "reference_scores/hunyuan3d",
    "TRELLIS.2": "reference_scores/vanilla_trellis2",
    "SymTRELLIS (Pred.)": (
        "experiments/score_shape_s114514_ns0.2_gs0.4_gd0.3_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_sparse_old_pred_sym_old_pred"
    ),
    "SymTRELLIS (GT)": (
        "experiments/score_shape_s114514_ns0.2_gs0.4_gd0.3_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_sparse_old_gt_sym_old_gt"
    ),
    "Closest-point Average (GT)": (
        "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_closest_point_average"
    ),
    "Sector Replication (GT)": (
        "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_sector_replication"
    ),
    "Voxel Majority (GT)": (
        "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_voxel_majority"
    ),
    "New SS Only (Pred.)": (
        "experiments/score_shape_s114514_ns0.0_gs0.0_gd0.0_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_ss_s114514_ns0.2_gs0.9_gd0.3_"
        "m_trellis2_sparse_structure_neighbor_graph_finetune_sym_old_pred_vc0_sym_gt"
    ),
    "New SS Only (GT)": (
        "experiments/score_shape_s114514_ns0.0_gs0.0_gd0.0_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_ss_s114514_ns0.2_gs0.9_gd0.3_"
        "m_trellis2_sparse_structure_neighbor_graph_finetune_sym_gt_vc0_sym_gt"
    ),
    "New SS + New Shape (Pred.)": (
        "experiments/score_shape_s114514_ns0.2_gs0.4_gd0.3_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_ss_s114514_ns0.2_gs0.9_gd0.3_"
        "m_trellis2_sparse_structure_neighbor_graph_finetune_sym_old_pred_vc0_sym_old_pred"
    ),
    "New SS + New Shape (GT)": (
        "experiments/score_shape_s114514_ns0.2_gs0.4_gd0.3_"
        "m_trellis2_shape_neighbor_graph_pretrain_ss_ss_s114514_ns0.2_gs0.9_gd0.3_"
        "m_trellis2_sparse_structure_neighbor_graph_finetune_sym_gt_vc0_sym_gt"
    ),
}

main_models = [
    "TripoSG",
    "Hunyuan3D-2.1",
    "TRELLIS.2",
    "SymTRELLIS (Pred.)",
    "SymTRELLIS (GT)",
    "Closest-point Average (GT)",
    "Sector Replication (GT)",
    "Voxel Majority (GT)",
]

ablation_models = [
    "TRELLIS.2",
    "SymTRELLIS (Pred.)",
    "SymTRELLIS (GT)",
]

## Data loading

Ground-truth folds are reconstructed from the shape filenames using the same point-group convention as the evaluation workspace. Each score JSON is flattened into one row while retaining `eval_success` as the sole success criterion.

In [ ]:
def read_shape_folds(dataworkspace_dir):
    polyhedral_folds = {"I": 5, "Ih": 5, "O": 4, "Oh": 4, "T": 3, "Td": 3, "Th": 3}
    rows = []

    for shape_path in sorted((dataworkspace_dir / "shapes").glob("*.glb")):
        shape_id_text, group_text, symmetry_axis = shape_path.stem.split("_")
        symmetry_group = group_text[0].upper() + group_text[1:]

        if symmetry_group in polyhedral_folds:
            symmetry_fold = polyhedral_folds[symmetry_group]
        elif symmetry_group == "S1":
            symmetry_fold = 1
        else:
            order = int("".join(character for character in symmetry_group if character.isdigit()))
            symmetry_fold = order // 2 if symmetry_group.startswith("S") else order

        rows.append(
            {
                "shape_id": int(shape_id_text),
                "symmetry_group": symmetry_group,
                "symmetry_axis": symmetry_axis,
                "gt_fold": symmetry_fold,
            }
        )

    return pd.DataFrame(rows).set_index("shape_id")


def read_trial(trial_name, relative_folder, dataworkspace_dir, metadata, shape_folds):
    rows = []

    for score_path in sorted((dataworkspace_dir / relative_folder).glob("*.json")):
        with score_path.open() as file:
            score = json.load(file)

        row = {
            "trial": trial_name,
            "idx": score_path.stem,
            "score_present": True,
            "eval_success": score["eval_success"],
        }

        if score["eval_success"]:
            symmetry = score["self_symm_result"]
            reconstruction = score["reconstruction_result"]
            row.update(
                {
                    "pred_fold": score["pred_fold"],
                    "max_sd": symmetry["max_sd"],
                    "mean_sd": symmetry["mean_sd"],
                    "max_bad_rate_001": symmetry["max_bad_rate"]["0.01"],
                    "max_bad_rate_003": symmetry["max_bad_rate"]["0.03"],
                    "max_bad_rate_01": symmetry["max_bad_rate"]["0.1"],
                    "mean_bad_rate_001": symmetry["mean_bad_rate"]["0.01"],
                    "mean_bad_rate_003": symmetry["mean_bad_rate"]["0.03"],
                    "mean_bad_rate_01": symmetry["mean_bad_rate"]["0.1"],
                    "cd_recon": reconstruction["cd_recon"],
                }
            )

        rows.append(row)

    trial_scores = pd.DataFrame(rows)
    trial_scores = metadata[["idx", "shape_id", "view_id"]].merge(
        trial_scores, on="idx", how="left"
    )
    trial_scores["trial"] = trial_name

    return trial_scores.merge(shape_folds[["gt_fold"]], on="shape_id", how="left")


def summarize_trials(scores, selected_models):
    selected_scores = scores[scores["trial"].isin(selected_models)]
    success_matrix = (
        selected_scores.assign(success=selected_scores["eval_success"].eq(True))
        .pivot(index="idx", columns="trial", values="success")
        .reindex(columns=selected_models, fill_value=False)
        .fillna(False)
    )
    common_success_ids = success_matrix.index[success_matrix.all(axis=1)]
    common_scores = selected_scores[selected_scores["idx"].isin(common_success_ids)].copy()
    common_scores["fold_accuracy"] = common_scores["pred_fold"].eq(common_scores["gt_fold"])

    metric_columns = [
        "max_sd",
        "mean_sd",
        "max_bad_rate_001",
        "max_bad_rate_003",
        "max_bad_rate_01",
        "mean_bad_rate_001",
        "mean_bad_rate_003",
        "mean_bad_rate_01",
        "fold_accuracy",
        "cd_recon",
    ]
    summary = (
        common_scores.groupby("trial", sort=False)[metric_columns]
        .mean()
        .reindex(selected_models)
    )
    summary["sample_count"] = common_scores.groupby("trial").size().reindex(selected_models)

    scaled_columns = [column for column in metric_columns if column != "fold_accuracy"]
    summary[scaled_columns] *= 1000
    summary["fold_accuracy"] *= 100

    return common_success_ids, summary

In [ ]:
metadata = pd.read_csv(dataworkspace_dir / "metadata.csv", dtype={"idx": str})
shape_folds = read_shape_folds(dataworkspace_dir)
selected_models = list(dict.fromkeys(main_models + ablation_models))
scores = pd.concat(
    [
        read_trial(
            trial_name,
            model_folders[trial_name],
            dataworkspace_dir,
            metadata,
            shape_folds,
        )
        for trial_name in selected_models
    ],
    ignore_index=True,
)

source_summary = (
    scores.assign(
        json_count=scores["score_present"].eq(True),
        successful=scores["eval_success"].eq(True),
        failed=scores["eval_success"].eq(False),
    )
    .groupby("trial", sort=False)[["json_count", "successful", "failed"]]
    .sum()
    .reindex(selected_models)
)
source_summary.insert(0, "relative_folder", [model_folders[name] for name in selected_models])
source_summary.index.name = "Trial"
source_summary

## Main comparison

All metrics except fold accuracy are reported in $\times 10^3$. Fold accuracy is a percentage.

In [ ]:
main_success_ids, main_summary = summarize_trials(scores, main_models)

main_subset = pd.DataFrame(
    {
        "Common-success samples": [len(main_success_ids)],
        "Total samples": [len(metadata)],
        "Coverage (%)": [100 * len(main_success_ids) / len(metadata)],
    },
    index=["Main table"],
)
display(main_subset.style.format({"Coverage (%)": "{:.2f}"}))

main_table = main_summary.rename(
    columns={
        "max_sd": "Max SD",
        "mean_sd": "Mean SD",
        "max_bad_rate_001": "Max Err. @ .01",
        "max_bad_rate_003": "Max Err. @ .03",
        "max_bad_rate_01": "Max Err. @ .1",
        "mean_bad_rate_001": "Mean Err. @ .01",
        "mean_bad_rate_003": "Mean Err. @ .03",
        "mean_bad_rate_01": "Mean Err. @ .1",
        "fold_accuracy": "Fold Acc. (%)",
        "cd_recon": "CD",
    }
)[
    [
        "Max SD",
        "Mean SD",
        "Max Err. @ .01",
        "Max Err. @ .03",
        "Max Err. @ .1",
        "Mean Err. @ .01",
        "Mean Err. @ .03",
        "Mean Err. @ .1",
        "Fold Acc. (%)",
        "CD",
    ]
]
main_table.index.name = "Method"
display(main_table.style.format("{:.3f}"))

## Ablation

The ablation table computes a separate common-success subset from only the trials displayed below. All reported metrics are in $\times 10^3$.

In [ ]:
ablation_success_ids, ablation_summary = summarize_trials(scores, ablation_models)

ablation_subset = pd.DataFrame(
    {
        "Common-success samples": [len(ablation_success_ids)],
        "Total samples": [len(metadata)],
        "Coverage (%)": [100 * len(ablation_success_ids) / len(metadata)],
    },
    index=["Ablation table"],
)
display(ablation_subset.style.format({"Coverage (%)": "{:.2f}"}))

ablation_table = ablation_summary.rename(
    columns={
        "max_bad_rate_003": "Max Err. @ .03",
        "mean_bad_rate_003": "Mean Err. @ .03",
        "cd_recon": "CD",
    }
)[["Max Err. @ .03", "Mean Err. @ .03", "CD"]]
ablation_table.index.name = "Method"
display(ablation_table.style.format("{:.3f}"))

## Consistency checks

In [ ]:
selected_for_tables = main_models + ablation_models
checks = pd.Series(
    {
        "Main rows share one sample count": main_summary["sample_count"].nunique() == 1,
        "Ablation rows share one sample count": ablation_summary["sample_count"].nunique() == 1,
        "Main and ablation subsets were computed separately": main_success_ids is not ablation_success_ids,
        "No new-SS trial is selected": not any(name.startswith("New SS") for name in selected_for_tables),
    },
    name="Passed",
)
checks.to_frame()